# TRL Trainer

In [1]:
import os
os.environ["CC"] = os.path.expanduser("~/gcc_wrapper")
os.environ.pop("GCC_EXEC_PREFIX", None)


'/usr/lib/gcc/x86_64-linux-gnu/13/:/usr/libexec/gcc/'

In [2]:
# if needed, or on terminal for lighteval as well: export CUDA_VISIBLE_DEVICES=7
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Replace 2 with the ID of the free GPU
os.environ["CUDA_VISIBLE_DEVICES"]

'0'

In [3]:
# Load model and tokenizer directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B-Base",     
                                          torch_dtype="auto",
                                          device_map="auto", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B-Base",
                                            torch_dtype="auto",
                                            device_map="auto", trust_remote_code=True)

In [4]:
import torch
torch.cuda.is_available()

True

In [5]:
model.get_memory_footprint()/1e9 # GB

1.192100096

In [6]:
"""
from datasets import Dataset

data = [
    {
        "prompt": "What is the capital of France?",
        "chosen": "Paris is the capital of France.",
        "rejected": "Berlin is the capital of France."
    },
    {
        "prompt": "What is 2 + 2?",
        "chosen": "2 + 2 equals 4.",
        "rejected": "2 + 2 equals 5."
    }
]

dataset = Dataset.from_list(data)
"""

'\nfrom datasets import Dataset\n\ndata = [\n    {\n        "prompt": "What is the capital of France?",\n        "chosen": "Paris is the capital of France.",\n        "rejected": "Berlin is the capital of France."\n    },\n    {\n        "prompt": "What is 2 + 2?",\n        "chosen": "2 + 2 equals 4.",\n        "rejected": "2 + 2 equals 5."\n    }\n]\n\ndataset = Dataset.from_list(data)\n'

In [7]:
from datasets import load_dataset
#ppdataset = datasets.load_dataset('json', data_files = '/home/project-m2-2025-intelligent-agents/data/pmp_dataset.jsonl')
#m1_dpo = load_dataset("ciacco/MNLP_M2_dpo_dataset", split="train")
#m1_dpo = load_dataset("ciacco/m1_dpo_filtered_random", split="train")
#m1_dpo = load_dataset("ciacco/m1_dpo_filtered_highest_total", split="train")
m1_dpo = load_dataset("ciacco/m1_dpo_juicy_mix", split="train")
m1_dpo

README.md:   0%|          | 0.00/444 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/79.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/40630 [00:00<?, ? examples/s]

Dataset({
    features: ['source', 'chosen', 'rejected', 'prompt', 'question'],
    num_rows: 40630
})

In [ ]:
m1_dpo = m1_dpo.

In [8]:
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layernorm): Qwe

## Vanilla DPO Trainer

In [9]:
from transformers import TrainingArguments
from trl import DPOTrainer, DPOConfig

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,
    args = DPOConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 2,
        warmup_ratio = 0.1,
        num_train_epochs = 1,
        learning_rate = 5e-7,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.0,
        lr_scheduler_type = "cosine",
        seed = 42,
        output_dir = "outputs/m1_highest_total",
        beta = 0.02,
        report_to = "none", # Use this for WandB etc
        max_length = 2048,
        max_prompt_length = 1024,
        #save_total_limit = ,
        gradient_checkpointing= True,
        save_steps=500,
    ),
    train_dataset = m1_dpo,#['train'],
    # eval_dataset = raw_datasets["test"],
    processing_class = tokenizer,
)

In [10]:
dpo_trainer.train()

Step,Training Loss
10,0.688300
20,0.686500
30,0.691600
40,0.691800
50,0.699800
60,0.692600
70,0.694300
80,0.693700
90,0.693600
100,0.690000


TrainOutput(global_step=3792, training_loss=0.6919692156184072, metrics={'train_runtime': 2485.5872, 'train_samples_per_second': 3.051, 'train_steps_per_second': 1.526, 'total_flos': 0.0, 'train_loss': 0.6919692156184072, 'epoch': 1.0})

In [11]:
!pwd

/home/project-m3-2025-intelligent-agents/code/train_dpo


In [18]:
# Load model and tokenizer directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer_ckpt = AutoTokenizer.from_pretrained("/home/project-m3-2025-intelligent-agents/code/train_dpo/outputs/m1_highest_total/checkpoint-1500",     
                                          torch_dtype="auto",
                                          device_map="auto")
model_ckpt = AutoModelForCausalLM.from_pretrained("/home/project-m3-2025-intelligent-agents/code/train_dpo/outputs/m1_highest_total/checkpoint-1500",
                                            torch_dtype="auto",
                                            device_map="auto")

In [19]:
#from huggingface_hub import notebook_login
#notebook_login()

In [20]:
model_ckpt.push_to_hub("ciacco/MNLP_M3_dpo_model_m1_highest_total_1500", private=True)
tokenizer_ckpt.push_to_hub("ciacco/MNLP_M3_dpo_model_m1_highest_total_1500", private=True)

Uploading...:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Uploading...:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/ciacco/MNLP_M3_dpo_model_m1_highest_total_1500/commit/0a41c447533dc2ff5e1f7a2c67cc2dfefe7b9a79', commit_message='Upload tokenizer', commit_description='', oid='0a41c447533dc2ff5e1f7a2c67cc2dfefe7b9a79', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ciacco/MNLP_M3_dpo_model_m1_highest_total_1500', endpoint='https://huggingface.co', repo_type='model', repo_id='ciacco/MNLP_M3_dpo_model_m1_highest_total_1500'), pr_revision=None, pr_num=None)

In [13]:
model.push_to_hub("ciacco/MNLP_M3_dpo_model_m1_highest_total", private=True)
tokenizer.push_to_hub("ciacco/MNLP_M3_dpo_model_m1_highest_total", private=True)

Uploading...:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

Uploading...:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/ciacco/MNLP_M3_dpo_model_m1_highest_total/commit/8b2c27ee95d9d5cc2ef8b8e344fffcbe4b4feb91', commit_message='Upload tokenizer', commit_description='', oid='8b2c27ee95d9d5cc2ef8b8e344fffcbe4b4feb91', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ciacco/MNLP_M3_dpo_model_m1_highest_total', endpoint='https://huggingface.co', repo_type='model', repo_id='ciacco/MNLP_M3_dpo_model_m1_highest_total'), pr_revision=None, pr_num=None)

# Unsloth Trainer

In [ ]:
import os
os.environ["CC"] = os.path.expanduser("~/gcc_wrapper")
os.environ.pop("GCC_EXEC_PREFIX", None)


In [ ]:
#import os
#del os.environ["GCC_EXEC_PREFIX"]
#os.environ["CC"] = "/home/gcc_wrapper"

In [11]:
from datasets import load_dataset
from unsloth import FastLanguageModel
from transformers import AutoTokenizer, AutoModelForCausalLM

# load Qwen3-Base
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen3-0.6B-Base",
    full_finetuning = True, # We have full finetuning now!
)

ModuleNotFoundError: No module named 'unsloth'

In [ ]:
from datasets import Dataset

data = [
    {
        "prompt": "What is the capital of France?",
        "chosen": "Paris is the capital of France.",
        "rejected": "Berlin is the capital of France."
    },
    {
        "prompt": "What is 2 + 2?",
        "chosen": "2 + 2 equals 4.",
        "rejected": "2 + 2 equals 5."
    }
]

dataset = Dataset.from_list(data)

In [ ]:
# One must patch the DPO Trainer first!
from unsloth import PatchDPOTrainer

PatchDPOTrainer()

In [ ]:
from transformers import TrainingArguments
from trl import DPOTrainer, DPOConfig

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,
    args = DPOConfig(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        num_train_epochs = 3,
        learning_rate = 5e-6,
        logging_steps = 1,
        optim = "sgd",
        weight_decay = 0.0,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        beta = 0.1,
        report_to = "none", # Use this for WandB etc
        max_length = 1024,
        max_prompt_length = 512,
    ),
    train_dataset = dataset,
    # eval_dataset = raw_datasets["test"],
    processing_class = tokenizer,

)

In [ ]:
dpo_trainer.train()